# 22 — Merkle Tree (Amazon FAR style)

A hash tree over data blocks — the integrity/sync primitive behind git, rsync, and blockchains. Parts
1-3 build the core (concurrency at Part 3); Parts 4-5 are stretch tasks (inclusion proofs, then
parallel leaf hashing). Fill stubs, run each test cell, peek at the collapsed `ref_` solutions only
after trying.

`_H(x)` (provided) is a stable SHA-256 hex hash. Odd levels duplicate the last node (a common scheme).

---

## Part 1 — Merkle root

`merkle_root(blocks) -> str`: hash each block into a leaf, then repeatedly hash adjacent pairs
(`_H(left + right)`) up to a single root. If a level has an odd count, duplicate its last node. Empty
input hashes to `_H(b"")`; a single block's root is just its leaf hash.

**Lock down:** any change to any block changes the root (tamper-evidence); the tree is balanced by
duplicating odd tails.

In [ ]:
import hashlib


def _H(x):
    if isinstance(x, str):
        x = x.encode()
    return hashlib.sha256(x).hexdigest()


def merkle_root(blocks):
    raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    r = merkle_root(["a", "b", "c", "d"])
    assert isinstance(r, str) and len(r) == 64
    assert merkle_root(["a", "b", "c", "d"]) == r                  # deterministic
    assert merkle_root(["a", "b", "c", "X"]) != r                  # tamper changes the root
    assert merkle_root(["solo"]) == _H("solo")                     # single block
    print("Part 1 OK")

_t1()

## Part 2 — Full tree & diff

- `merkle_tree(blocks) -> list[list]`: all levels, `level[0]` = leaf hashes … last level = `[root]`.
- `diff(tree_a, tree_b) -> list[int]`: the leaf **indices** where two trees differ. Short-circuit:
  if the roots match, return `[]` (the cheap "are they identical?" check).

**Lock down:** equal roots ⇒ identical data (the whole point — one hash comparison replaces comparing
every block); otherwise report which leaves changed.

In [ ]:
def merkle_tree(blocks):
    raise NotImplementedError


def diff(tree_a, tree_b):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    A = merkle_tree(["a", "b", "c", "d"])
    assert A[0] == [_H(x) for x in ("a", "b", "c", "d")] and len(A[-1]) == 1
    assert diff(A, merkle_tree(["a", "b", "c", "d"])) == []         # identical -> no diff
    assert diff(A, merkle_tree(["a", "X", "c", "d"])) == [1]
    assert diff(A, merkle_tree(["a", "X", "c", "Y"])) == [1, 3]
    print("Part 2 OK")

_t2()

## Part 3 — Concurrent leaf hashing

`concurrent_merkle_root(blocks, num_workers) -> str`: hash the leaves across `num_workers` threads
(into their correct slots), then combine into the root. Must equal `merkle_root`.

**Discuss:** leaves are independent so workers write **distinct** indices (no shared-state race); the
combine step is sequential; in CPython threads don't speed up the hashing itself (GIL) — Part 5 uses
processes for that. Writing distinct list indices from threads is safe; a shared counter wouldn't be.

In [ ]:
import queue
import threading


def concurrent_merkle_root(blocks, num_workers):
    raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    blocks = ["block%d" % i for i in range(100)]
    for _ in range(3):                                              # repeat to surface races
        assert concurrent_merkle_root(blocks, 8) == merkle_root(blocks)
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Inclusion proofs

A light client can verify a block is in the tree without the whole tree, using a **Merkle proof** (the
sibling hash at each level).
- `merkle_proof(blocks, index) -> list`: the sibling hashes from leaf to root.
- `verify_proof(leaf_block, index, proof, root) -> bool`: recompute the root from the leaf + proof and
  compare. A tampered block fails.

**Discuss:** O(log n) proof size; left/right ordering at each step (driven by the index bit); this is
how SPV wallets / certificate transparency / content-addressed sync verify membership cheaply.

In [ ]:
def merkle_proof(blocks, index):
    raise NotImplementedError


def verify_proof(leaf_block, index, proof, root):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    blocks = ["a", "b", "c", "d"]
    root = merkle_root(blocks)
    for i in range(4):
        assert verify_proof(blocks[i], i, merkle_proof(blocks, i), root)
    assert not verify_proof("TAMPERED", 1, merkle_proof(blocks, 1), root)
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel leaf hashing

`parallel_merkle_root(blocks, num_procs) -> str`: hash the leaves across processes with
`ProcessPoolExecutor` (worker `merkle_workers.hash_block`), then combine into the root. Must equal
`merkle_root`.

**Discuss:** leaf hashing is the CPU-bound, embarrassingly-parallel part; the tree combine is cheap
and sequential; the stable hash is what makes parent/workers agree.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import merkle_workers


def parallel_merkle_root(blocks, num_procs):
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    blocks = ["b%d" % i for i in range(50)]
    assert parallel_merkle_root(blocks, 4) == merkle_root(blocks)
    print("Part 5 OK")

_t5()

## Likely follow-ups
- O(log n) top-down diff (descend only where subtree hashes differ) instead of comparing all leaves.
- Sparse / binary Merkle tries for key-value authentication; Merkle Patricia tries (Ethereum).
- Consistency proofs (append-only logs / certificate transparency); handling non-power-of-two sizes.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import hashlib
import queue
import threading
from concurrent.futures import ProcessPoolExecutor
import merkle_workers


def _H(x):
    if isinstance(x, str):
        x = x.encode()
    return hashlib.sha256(x).hexdigest()


def _combine(level):
    while len(level) > 1:
        if len(level) % 2 == 1:
            level = level + [level[-1]]                 # duplicate odd tail
        level = [_H(level[i] + level[i + 1]) for i in range(0, len(level), 2)]
    return level[0]


def ref_merkle_root(blocks):
    if not blocks:
        return _H(b"")
    return _combine([_H(b) for b in blocks])


def ref_merkle_tree(blocks):
    if not blocks:
        return [[_H(b"")]]
    levels = [[_H(b) for b in blocks]]
    while len(levels[-1]) > 1:
        cur = levels[-1]
        if len(cur) % 2 == 1:
            cur = cur + [cur[-1]]
        levels.append([_H(cur[i] + cur[i + 1]) for i in range(0, len(cur), 2)])
    return levels


def ref_diff(tree_a, tree_b):
    if tree_a[-1] == tree_b[-1]:                        # roots equal -> identical
        return []
    a, b = tree_a[0], tree_b[0]
    n = max(len(a), len(b))
    return [i for i in range(n)
            if (a[i] if i < len(a) else None) != (b[i] if i < len(b) else None)]


def ref_concurrent_merkle_root(blocks, num_workers):
    if not blocks:
        return _H(b"")
    leaves = [None] * len(blocks)
    q = queue.Queue()
    for i, b in enumerate(blocks):
        q.put((i, b))
    for _ in range(num_workers):
        q.put(None)

    def worker():
        while True:
            item = q.get()
            if item is None:
                return
            i, b = item
            leaves[i] = _H(b)                           # distinct index -> no race

    threads = [threading.Thread(target=worker) for _ in range(num_workers)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return _combine(leaves)


def ref_merkle_proof(blocks, index):
    level = [_H(b) for b in blocks]
    proof, idx = [], index
    while len(level) > 1:
        if len(level) % 2 == 1:
            level = level + [level[-1]]
        proof.append(level[idx ^ 1])                    # sibling
        idx //= 2
        level = [_H(level[i] + level[i + 1]) for i in range(0, len(level), 2)]
    return proof


def ref_verify_proof(leaf_block, index, proof, root):
    h, idx = _H(leaf_block), index
    for sib in proof:
        h = _H(h + sib) if idx % 2 == 0 else _H(sib + h)
        idx //= 2
    return h == root


def ref_parallel_merkle_root(blocks, num_procs):
    if not blocks:
        return _H(b"")
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        leaves = list(ex.map(merkle_workers.hash_block, blocks))
    return _combine(leaves)


assert ref_merkle_root(["x"]) == _H("x")
_A = ref_merkle_tree(["a", "b", "c", "d"]); _B = ref_merkle_tree(["a", "b", "Z", "d"])
assert ref_diff(_A, _A) == [] and ref_diff(_A, _B) == [2]
assert ref_concurrent_merkle_root(["a", "b", "c"], 4) == ref_merkle_root(["a", "b", "c"])
_blocks = ["a", "b", "c", "d"]; _root = ref_merkle_root(_blocks)
assert ref_verify_proof("c", 2, ref_merkle_proof(_blocks, 2), _root)
assert not ref_verify_proof("nope", 2, ref_merkle_proof(_blocks, 2), _root)
assert ref_parallel_merkle_root(["a", "b", "c", "d", "e"], 4) == ref_merkle_root(["a", "b", "c", "d", "e"])
print("reference solutions OK")